# USB6451 periodic sinewave demo (no UI)

This notebook starts and stops continuous sine output on `AO` using `USB6451.start_continuous_sine_output(...)`.

## Important: non-integer frequency behavior

- In periodic regenerate mode, one period is stored and repeated.
- Output frequency follows `sample_rate / samples_per_period`.
- If `samples_per_period` is `None`, the class uses `round(sample_rate / frequency)`.
- So requested frequency can be adjusted to the nearest realizable periodic value.
- If `samples_per_period` is provided, it must match the requested frequency or an error is raised.

Example: with `sample_rate = 10000` and requested `frequency = 17`, periodic mode gives `10000 / 588 = 17.0068 Hz`.

Run cells top-to-bottom:
1. Import/setup
2. Set parameters
3. Start output
4. Stop output when done

## 1. Import and setup


In [3]:
import atexit
import sys
from pathlib import Path


def add_repo_to_path() -> None:
    """Ensure repository root is importable in this notebook session."""
    cwd = Path.cwd()
    if (cwd / "USB6451" / "USB6451.py").exists():
        sys.path.insert(0, str(cwd))
        return

    candidate = cwd.parent.parent
    if (candidate / "USB6451" / "USB6451.py").exists():
        sys.path.insert(0, str(candidate))
        return

    raise RuntimeError("Could not find repository root containing USB6451/USB6451.py")


add_repo_to_path()
from USB6451.USB6451 import USB6451

daq = USB6451()
atexit.register(daq.close)
print("USB6451 object created.")

USB6451 object created.


## 2. Set parameters


In [17]:
# Set parameters here.
device = "Dev1"
ao_channel = "ao0"
frequency = 35.0
amplitude = 1.0
offset = 0.0
sample_rate = 170000.0
samples_per_period = None  # Set int value (for example 1000) or keep None.
min_voltage = -10.0
max_voltage = 10.0
allow_regen = True

validated = daq.get_continuous_sine_output_config(
    device=device,
    ao_channel=ao_channel,
    frequency=frequency,
    amplitude=amplitude,
    offset=offset,
    sample_rate=sample_rate,
    samples_per_period=samples_per_period,
    min_voltage=min_voltage,
    max_voltage=max_voltage,
)

print("Parameters validated.")
print(f"Requested frequency: {validated.requested_frequency:.9g} Hz")
print(f"Actual frequency:    {validated.actual_frequency:.9g} Hz")
print(f"samples_per_period:  {validated.samples_per_period}")


Parameters validated.
Requested frequency: 35 Hz
Actual frequency:    35.0010294 Hz
samples_per_period:  4857


## 3. Start output


In [20]:
# Start output using validated parameters from step 2.
actual_frequency = daq.start_continuous_sine_output(
    device=validated.device,
    ao_channel=validated.ao_channel,
    frequency=validated.actual_frequency,
    amplitude=validated.amplitude,
    offset=validated.offset,
    sample_rate=validated.sample_rate,
    samples_per_period=validated.samples_per_period,
    min_voltage=validated.min_voltage,
    max_voltage=validated.max_voltage,
    allow_regen=allow_regen,
)

print("Output started.")
print(f"Running at {actual_frequency:.9g} Hz")
print("Run the stop cell when you want to stop output.")


Output started.
Running at 35.0010294 Hz
Run the stop cell when you want to stop output.


## 4. Stop output


In [21]:
# Stop output.
daq.stop_output()
print("Output stopped.")

Output stopped.


## 5. Set non-regen parameters

Use this mode when `sample_rate / frequency` is not an integer and you want exact requested frequency.

In [1]:
# Set non-regen parameters here.
non_regen_device = "Dev1"
non_regen_ao_channel = "ao0"
non_regen_frequency = 17.0
non_regen_amplitude = 1.0
non_regen_offset = 0.0
non_regen_sample_rate = 10000.0
non_regen_chunk_samples = 1000
non_regen_run_seconds = 3.0
non_regen_min_voltage = -10.0
non_regen_max_voltage = 10.0

divider = non_regen_sample_rate / non_regen_frequency
print(f"Requested frequency: {non_regen_frequency:.9g} Hz")
print(f"Sample rate: {non_regen_sample_rate:.9g} S/s")
print(f"sample_rate / frequency = {divider:.12f}")
print("Non-integer divider => use non-regen streaming.")


Requested frequency: 17 Hz
Sample rate: 10000 S/s
sample_rate / frequency = 588.235294117647
Non-integer divider => use non-regen streaming.


## 6. Start and stream non-regen output

In [ ]:
# Start non-regen output and keep feeding chunks for a fixed time.
import time

daq.stop_output()
actual_non_regen_sample_rate = daq.start_continuous_sine_output_non_regen(
    device=non_regen_device,
    ao_channel=non_regen_ao_channel,
    frequency=non_regen_frequency,
    amplitude=non_regen_amplitude,
    offset=non_regen_offset,
    sample_rate=non_regen_sample_rate,
    chunk_samples=non_regen_chunk_samples,
    min_voltage=non_regen_min_voltage,
    max_voltage=non_regen_max_voltage,
)

chunks_written = 1  # First chunk is written by start_continuous_sine_output_non_regen.
end_time = time.time() + non_regen_run_seconds
while time.time() < end_time:
    daq.write_sine_chunk_non_regen(chunk_samples=non_regen_chunk_samples)
    chunks_written += 1

print("Non-regen output done.")
print(f"Actual sample rate: {actual_non_regen_sample_rate:.9g} S/s")
print(f"Chunks written: {chunks_written}")
print(f"Samples per chunk: {non_regen_chunk_samples}")


## 7. Stop non-regen output

In [ ]:
# Stop non-regen output.
daq.stop_output()
print("Non-regen output stopped.")
